# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# Explore the available record sets in the dataset by their @id
record_sets = list(dataset.record_sets.keys())
print(f"Record Sets (@id): {record_sets}")
# For each record set, print its fields and columns by @id
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"\nRecord Set: {rs_id}")
    # List fields (by @id)
    field_ids = [field['@id'] for field in rs.get('field', [])]
    print(f"  Fields (@id): {field_ids}")
    # List columns for each field (by @id)
    if 'field' in rs:
        for field in rs['field']:
            col_ids = []
            if 'column' in field:
                columns = field['column']
                if isinstance(columns, list):
                    col_ids = [col['@id'] if isinstance(col, dict) and '@id' in col else col for col in columns]
                elif isinstance(columns, dict):
                    col_ids = [columns['@id']]
                else:
                    col_ids = [columns]
            print(f"    Field {field['@id']} columns: {col_ids}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Record set and field `@id`s are used for clarity and reproducibility.

In [ ]:
# List all available record sets by @id
record_sets = list(dataset.record_sets.keys())
dataframes = {}

# For demonstration, let's extract all record sets
for record_set_id in record_sets:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"  No records found for {record_set_id}")

# If there are no record sets, try loading the default one (sometimes there can be a single record set key like 'cr:RecordSet')
if not dataframes:
    print("No record sets found in the Croissant metadata. Try loading the default records.")
    try:
        records = list(dataset.records())
        if records:
            df = pd.DataFrame(records)
            dataframes['default'] = df
            print(f"  Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print('No records found in the dataset.')
    except Exception as e:
        print(f'Error extracting records: {e}')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set for EDA
if len(dataframes) > 0:
    # Select the first available DataFrame
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()
    print(f"Using record set: {record_set_id}")

    # List numeric fields to choose from
    numeric_columns = df.select_dtypes(include=['float', 'int']).columns.tolist()
    print(f"Numeric columns: {numeric_columns}")

    # For illustration, select a numeric field (use the first found)
    if numeric_columns:
        numeric_field = numeric_columns[0]
        # Define a threshold (use 10 unless values are much larger)
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (total: {len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a likely group field (if available, e.g. a category/text field)
        group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        for group_field in group_candidates:
            if df[group_field].nunique() < 20:  # reasonable groupings
                print(f"Grouped by {group_field} (mean {numeric_field}):")
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                display(grouped_df.head())
                break
        else:
            print("No group field with <20 unique values found for grouping.")
    else:
        print('No numeric columns found for EDA.')
else:
    print('No DataFrames available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'df' in locals() and not df.empty and numeric_columns:
    # Plot histogram of the selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot between two numeric fields if more than 1
    if len(numeric_columns) > 1:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=df[numeric_columns[0]], y=df[numeric_columns[1]])
        plt.title(f"{numeric_columns[0]} vs {numeric_columns[1]}")
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.show()
else:
    print('No numeric columns to visualize.')

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset via Croissant, inspected its record sets and fields, extracted records, performed basic exploratory analysis and data transformations, and visualized the main numeric attributes. You can extend this notebook for advanced modeling and downstream analysis using the `mlcroissant` paradigm.